In [1]:
import warnings

# 모든 경고 메시지 무시
warnings.filterwarnings('ignore')

In [2]:
import rustima
import numpy as np

print(rustima.version())                              # "0.1.0"
y = np.random.randn(100).cumsum()
r = rustima.sarimax_fit(y, order=(1, 1, 1), seasonal=(0, 0, 0, 0))
print(f"converged={r['converged']}, AIC={r['aic']:.2f}")

0.1.0
converged=True, AIC=281.03


In [3]:
import os, time, threading, gc, tracemalloc, warnings
import numpy as np
import polars as pl
import psutil

In [4]:
# 데이터 로드 (파주 지점, 최근 60일 = 1,440h)
df = pl.read_csv("/Users/lab2/Documents/GitHub/Rust-python-arima2/rustima/0_Example_cy/total_heat_demand_dwh.csv")
df_paju = (
    df.filter(pl.col('branch_id') == '파주')
      .with_columns(pl.col('date').str.to_datetime())
      .sort('date')
)

In [5]:
len(df_paju)

26279

In [15]:
# df_paju → numpy array (y)
y = df_paju['heat_demand'].to_numpy().astype(np.float64)
print(f"y length: {len(y)}")

y length: 26279


# rustima vs pmdarima 비교

- `seasonal × stepwise × trend` = **2 × 2 × 4 = 16 조합**, rustima vs pmdarima 2개로 **총 32런** 자동 실행
- s=24, max_p=5, max_q=5, max_P=2, max_Q=2, criterion="aic" 고정
- 각 런마다 50ms 간격으로 RSS를 샘플링한 **메모리 타임라인(JSON)** 과 `trace=True` **stdout 로그(.log)** 를 저장

In [16]:
# ======================================================================
# sweep-clean — sweep-run 실행 전 기존 산출물 정리
#
# MODE 를 바꿔서 원하는 수준의 정리를 실행하세요.
#   "none"  → 아무것도 안 함 (현재 상태만 출력)
#   "pkl"   → sweep_*/y.pkl 만 삭제 (sweep-run 이 y 를 새로 pickle)
#   "dir"   → sweep_* 디렉터리 전체 삭제 (완전 초기화; RESUME 무효)
#   "latest"→ 가장 최근 sweep_* 디렉터리만 통째로 삭제
# ======================================================================
import shutil
from pathlib import Path

MODE = "pkl"       # "none" | "pkl" | "dir" | "latest"
DRY_RUN = False    # True 이면 삭제 대상만 표시

base = Path("bench_results")
if not base.exists():
    print("bench_results 디렉터리 없음 — 정리할 것 없음")
else:
    sweep_dirs = sorted(base.glob("sweep_*"))
    print(f"현재 sweep 디렉터리: {len(sweep_dirs)}개")
    for d in sweep_dirs:
        csv_p = d / "sweep_results.csv"
        pkl_p = d / "y.pkl"
        n_rows = 0
        if csv_p.exists():
            try:
                n_rows = sum(1 for _ in open(csv_p, encoding="utf-8")) - 1
            except Exception:
                pass
        print(f"  {d.name}  rows={n_rows:3d}  y.pkl={'O' if pkl_p.exists() else '-'}")

    def _rm_file(p):
        print(f"  rm  {p}")
        if not DRY_RUN:
            try: p.unlink()
            except OSError as e: print(f"     (failed: {e})")

    def _rm_tree(p):
        print(f"  rmtree  {p}")
        if not DRY_RUN:
            try: shutil.rmtree(p)
            except OSError as e: print(f"     (failed: {e})")

    if MODE == "none":
        print("\nMODE=none — 건너뜀")
    elif MODE == "pkl":
        if "y" not in globals():
            print("⚠️  [MODE=pkl] 커널에 y 가 없습니다. y.pkl 을 지우면 "
                  "RESUME 시 sweep-run 이 RuntimeError 로 실패합니다.")
            print("    먼저 y 정의 셀을 실행한 뒤 다시 시도하거나, "
                  "전체 초기화가 목적이면 MODE=\"latest\" 또는 \"dir\" 을 쓰세요.")
        else:
            targets = [p for d in sweep_dirs for p in d.glob("*.pkl")]
            print(f"[MODE=pkl] {len(targets)}개 pkl 삭제 (DRY_RUN={DRY_RUN})")
            for p in targets:
                _rm_file(p)
    elif MODE == "dir":
        print(f"\n[MODE=dir] {len(sweep_dirs)}개 sweep 디렉터리 전체 삭제 (DRY_RUN={DRY_RUN})")
        for d in sweep_dirs:
            _rm_tree(d)
    elif MODE == "latest":
        if sweep_dirs:
            print(f"\n[MODE=latest] 가장 최근 1개 삭제 (DRY_RUN={DRY_RUN})")
            _rm_tree(sweep_dirs[-1])
        else:
            print("\n[MODE=latest] 대상 없음")
    else:
        raise ValueError(f"알 수 없는 MODE: {MODE!r}")

    if not DRY_RUN and MODE != "none":
        print(f"\n✅ 정리 완료. 남은 sweep 디렉터리: "
              f"{len(list(base.glob('sweep_*')))}개")

bench_results 디렉터리 없음 — 정리할 것 없음


In [17]:
# ======================================================================
# 전체 조합 스위프 — subprocess 격리 + Resume + 재시도
#
# 16 조합 × 2 라이브러리 = 32 런
# 조건: s=24, max_p=5, max_q=5, max_P=2, max_Q=2, criterion="aic"
# ======================================================================
import itertools
import subprocess
import pickle
import csv
import json
import time
import sys
import os
import textwrap
from pathlib import Path
from datetime import datetime

# ---- 설정 ------------------------------------------------------------
SEASONAL_OPTS = [(False, 0), (True, 24)]
STEPWISE_OPTS = [True, False]
TREND_OPTS = ["n", "c", "t", "ct"]
CRITERION = "aic"                          # 고정

MAX_P = 5
MAX_Q = 5
MAX_PP = 2   # max_P (seasonal)
MAX_QQ = 2   # max_Q (seasonal)

PER_RUN_TIMEOUT_S = 3600
MEM_INTERVAL_S = 0.05
MAX_ATTEMPTS = 1
RETRY_BACKOFF_S = 2.0
RESUME = True

combos = list(itertools.product(SEASONAL_OPTS, STEPWISE_OPTS, TREND_OPTS))
RUNNERS = ["rustima", "pmdarima"]
total_runs = len(combos) * len(RUNNERS)
print(f"조합 수: {len(combos)},  총 런: {total_runs}")

# ---- 출력 디렉터리 ---------------------------------------------------
base = Path("bench_results")
base.mkdir(exist_ok=True)

FIELDS = [
    "idx", "library", "seasonal", "s", "stepwise", "trend",
    "status", "order", "seasonal_order", "ic_value", "loglik_own",
    "time_s", "mem_baseline_mb", "mem_peak_mb", "mem_delta_mb",
    "attempts", "attempt_history",
    "error", "log_file", "mem_file",
]

OUT_DIR = None
if RESUME:
    for d in sorted(base.glob("sweep_*"), reverse=True):
        if (d / "sweep_results.csv").exists():
            OUT_DIR = d
            print(f"🔁 Resume — 기존 디렉터리: {OUT_DIR}")
            break
if OUT_DIR is None:
    STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUT_DIR = base / f"sweep_{STAMP}"
    print(f"🆕 신규 디렉터리: {OUT_DIR}")

LOG_DIR = OUT_DIR / "logs"
MEM_DIR = OUT_DIR / "mem_samples"
TMP_DIR = OUT_DIR / "_tmp"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MEM_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUT_DIR / "sweep_results.csv"
if not csv_path.exists():
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()

# ---- 완료 집합 -------------------------------------------------------
completed = set()
if csv_path.exists():
    with open(csv_path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if r.get("status"):
                try:
                    completed.add((int(r["idx"]), r["library"]))
                except (TypeError, ValueError):
                    pass
print(f"이미 완료: {len(completed)} / {total_runs}")

# ---- y pickle --------------------------------------------------------
Y_PKL = OUT_DIR / "y.pkl"
if "y" in dir():
    with open(Y_PKL, "wb") as f:
        pickle.dump(y, f)
    print(f"y pickled → {Y_PKL}  (len={len(y) if hasattr(y,'__len__') else '?'})")
elif Y_PKL.exists():
    print(f"⚠️  커널에 y 미정의. 기존 y.pkl 재사용: {Y_PKL}")
else:
    raise RuntimeError(
        f"y 가 커널에도 없고 {Y_PKL} 도 없습니다. 데이터 로딩 셀을 먼저 실행하세요.")

# ---- 워커 스크립트 ---------------------------------------------------
WORKER = OUT_DIR / "_worker.py"
WORKER.write_text(textwrap.dedent(r"""
    import sys, os, time, json, pickle, threading, traceback, gc
    (tag, lib, s, stepwise, criterion, trend, y_pkl, result_json,
     mem_json, log_txt, interval,
     max_p, max_q, max_pp, max_qq) = sys.argv[1:16]
    s = int(s); stepwise = stepwise == "1"; interval = float(interval)
    max_p = int(max_p); max_q = int(max_q)
    max_pp = int(max_pp); max_qq = int(max_qq)

    import psutil
    proc = psutil.Process(os.getpid())

    stop = threading.Event()
    samples = []
    baseline = proc.memory_info().rss / 1024 ** 2
    peak = [baseline]

    def _flush_mem():
        with open(mem_json, "w", encoding="utf-8") as f:
            json.dump({"tag": tag, "baseline_mb": baseline,
                       "peak_mb": peak[0], "delta_mb": peak[0] - baseline,
                       "interval_s": interval, "samples": samples,
                       "partial": not stop.is_set()}, f)

    def _sampler():
        t0 = time.perf_counter()
        samples.append((0.0, baseline))
        i = 0
        while not stop.is_set():
            m = proc.memory_info().rss / 1024 ** 2
            samples.append((time.perf_counter() - t0, m))
            if m > peak[0]: peak[0] = m
            i += 1
            if i % 10 == 0:
                try: _flush_mem()
                except Exception: pass
            stop.wait(interval)

    th = threading.Thread(target=_sampler, daemon=True)
    th.start()

    with open(y_pkl, "rb") as f:
        y = pickle.load(f)

    import io, contextlib
    buf = io.StringIO()
    status, err = "ok", ""
    order = seasonal_order = ic_val = loglik_own = None
    elapsed = None
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            t0 = time.perf_counter()
            if lib == "rustima":
                from rustima import auto_arima as rs_auto_arima
                res = rs_auto_arima(y, s=s, trend=trend, stepwise=stepwise,
                                    max_p=max_p, max_q=max_q,
                                    max_P=max_pp, max_Q=max_qq,
                                    trace=True, criterion=criterion)
                order = str(tuple(res.order))
                so = getattr(res, "seasonal_order", None)
                seasonal_order = str(tuple(so)) if so is not None else ""
                ic_val = getattr(res.result, criterion, None)
                loglik_own = (getattr(res.result, "loglik", None)
                              or getattr(res.result, "llf", None)
                              or getattr(res.result, "log_likelihood", None))
            else:
                import pmdarima as pm
                res = pm.auto_arima(
                    y, seasonal=(s > 0), m=s if s > 0 else 1, trend=trend,
                    stepwise=stepwise, suppress_warnings=True,
                    max_p=max_p, max_q=max_q,
                    max_P=max_pp, max_Q=max_qq,
                    information_criterion=criterion, trace=True)
                order = str(tuple(res.order))
                seasonal_order = str(tuple(res.seasonal_order))
                getter = getattr(res, criterion, None)
                ic_val = getter() if callable(getter) else None
                ar = getattr(res, "arima_res_", None)
                if ar is not None:
                    loglik_own = getattr(ar, "llf", None)
                if loglik_own is None:
                    llf_fn = getattr(res, "llf", None)
                    loglik_own = llf_fn() if callable(llf_fn) else getattr(res, "loglik", None)
            elapsed = time.perf_counter() - t0
    except Exception as e:
        status = "fail"
        err = f"{type(e).__name__}: {e}"
        traceback.print_exc(file=buf)

    stop.set(); th.join()
    _flush_mem()
    with open(log_txt, "w", encoding="utf-8") as f:
        f.write(buf.getvalue())
    with open(result_json, "w", encoding="utf-8") as f:
        json.dump({"status": status, "error": err,
                   "order": order or "", "seasonal_order": seasonal_order or "",
                   "ic_value": ic_val if isinstance(ic_val, (int, float)) else None,
                   "loglik_own": loglik_own if isinstance(loglik_own, (int, float)) else None,
                   "time_s": elapsed,
                   "mem_baseline_mb": baseline,
                   "mem_peak_mb": peak[0],
                   "mem_delta_mb": peak[0] - baseline}, f)
"""), encoding="utf-8")


def _run_one_attempt(cmd, result_json, log_path, tmp_log_path):
    """한 번의 subprocess 시도."""
    out = {"status": "", "error": "", "order": "", "seasonal_order": "",
           "ic_value": "", "loglik_own": "", "time_s": "",
           "mem_baseline_mb": "", "mem_peak_mb": "", "mem_delta_mb": ""}
    try:
        cp = subprocess.run(cmd, timeout=PER_RUN_TIMEOUT_S,
                            capture_output=True, text=True,
                            encoding="utf-8", errors="replace")
        if result_json.exists():
            d = json.loads(result_json.read_text(encoding="utf-8"))
            out["status"] = d.get("status", "ok")
            out["error"] = (d.get("error") or "")[:500]
            out["order"] = d.get("order", "")
            out["seasonal_order"] = d.get("seasonal_order", "")
            for src_k, dst_k, fmt in (
                ("ic_value", "ic_value", "{:.4f}"),
                ("loglik_own", "loglik_own", "{:.4f}"),
                ("time_s", "time_s", "{:.4f}"),
                ("mem_baseline_mb", "mem_baseline_mb", "{:.2f}"),
                ("mem_peak_mb", "mem_peak_mb", "{:.2f}"),
                ("mem_delta_mb", "mem_delta_mb", "{:.2f}"),
            ):
                v = d.get(src_k)
                out[dst_k] = fmt.format(v) if isinstance(v, (int, float)) else ""
            try: result_json.unlink()
            except OSError: pass
        else:
            out["status"] = "crashed"
            tail = (cp.stderr or cp.stdout or "").splitlines()[-5:]
            out["error"] = (" | ".join(tail))[:500]
            with open(tmp_log_path, "a", encoding="utf-8") as f:
                f.write(f"\n--- SUBPROCESS CRASH (exit={cp.returncode}) ---\n")
                f.write("STDOUT:\n" + (cp.stdout or "") + "\n")
                f.write("STDERR:\n" + (cp.stderr or "") + "\n")
    except subprocess.TimeoutExpired:
        out["status"] = "timeout"
        out["error"] = f"exceeded {PER_RUN_TIMEOUT_S}s"
        with open(tmp_log_path, "a", encoding="utf-8") as f:
            f.write(f"\n--- TIMEOUT after {PER_RUN_TIMEOUT_S}s ---\n")
    except Exception as e:
        out["status"] = "launch_fail"
        out["error"] = f"{type(e).__name__}: {e}"
    return out


# ---- 메인 루프 -------------------------------------------------------
sweep_t0 = time.perf_counter()
run_i = 0
for idx, ((seasonal, s), stepwise, trend) in enumerate(combos):
    for lib_name in RUNNERS:
        run_i += 1
        if (idx, lib_name) in completed:
            continue
        tag = f"{idx:02d}_{lib_name}_s{s}_sw{int(stepwise)}_{trend}"
        canonical_mem = MEM_DIR / f"{tag}.json"
        canonical_log = LOG_DIR / f"{tag}.log"

        print(f"\n{'=' * 80}")
        print(f"[{run_i:03d}/{total_runs}] {lib_name}  seasonal={seasonal}(s={s}) "
              f"stepwise={stepwise} trend={trend}")
        print("=" * 80)

        attempt_mem_paths = []
        attempt_log_paths = []
        attempt_histories = []
        final_out = None

        for attempt in range(1, MAX_ATTEMPTS + 1):
            tmp_mem = TMP_DIR / f"{tag}_try{attempt}.mem.json"
            tmp_log = TMP_DIR / f"{tag}_try{attempt}.log"
            tmp_result = TMP_DIR / f"{tag}_try{attempt}.result.json"

            cmd = [sys.executable, str(WORKER), tag, lib_name, str(s),
                   "1" if stepwise else "0", CRITERION, trend,
                   str(Y_PKL), str(tmp_result), str(tmp_mem), str(tmp_log),
                   str(MEM_INTERVAL_S),
                   str(MAX_P), str(MAX_Q), str(MAX_PP), str(MAX_QQ)]

            print(f"  ▶ attempt {attempt}/{MAX_ATTEMPTS}")
            out = _run_one_attempt(cmd, tmp_result, canonical_log, tmp_log)
            attempt_mem_paths.append(tmp_mem)
            attempt_log_paths.append(tmp_log)
            attempt_histories.append(out["status"])
            final_out = out

            if out["status"] == "ok":
                print(f"    → ok")
                break
            print(f"    → {out['status']}: {(out['error'] or '')[:140]}")
            if attempt < MAX_ATTEMPTS:
                time.sleep(RETRY_BACKOFF_S)

        final_mem = attempt_mem_paths[-1]
        final_log = attempt_log_paths[-1]
        try:
            if canonical_mem.exists(): canonical_mem.unlink()
            if final_mem.exists(): final_mem.replace(canonical_mem)
        except OSError: pass
        try:
            if canonical_log.exists(): canonical_log.unlink()
            if final_log.exists(): final_log.replace(canonical_log)
        except OSError: pass
        for mp, lp in zip(attempt_mem_paths[:-1], attempt_log_paths[:-1]):
            for p in (mp, lp):
                try:
                    if p.exists(): p.unlink()
                except OSError: pass

        row = {k: "" for k in FIELDS}
        row.update({
            "idx": idx, "library": lib_name,
            "seasonal": seasonal, "s": s, "stepwise": stepwise,
            "trend": trend,
            "log_file": str(canonical_log.relative_to(OUT_DIR)),
            "mem_file": str(canonical_mem.relative_to(OUT_DIR)),
            "attempts": len(attempt_histories),
            "attempt_history": ";".join(attempt_histories),
        })
        row.update(final_out)

        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDS).writerow(row)

        print(f"  ▣ final [{row['status']}] attempts={len(attempt_histories)} "
              f"history={row['attempt_history']}")
        if row["status"] == "ok":
            print(f"    order={row['order']} aic={row['ic_value']} "
                  f"loglik={row['loglik_own']} time={row['time_s']}s "
                  f"peak={row['mem_peak_mb']}MB")
        elif row["error"]:
            print(f"    err: {row['error'][:200]}")

        completed.add((idx, lib_name))

elapsed = time.perf_counter() - sweep_t0
print(f"\n✅ 세션 종료 — 누적 완료 {len(completed)}/{total_runs}, "
      f"이번 세션 {elapsed/60:.1f}분 소요")
print(f"   CSV : {csv_path}")
print(f"   로그 : {LOG_DIR}")
print(f"   메모리 : {MEM_DIR}")

조합 수: 16,  총 런: 32
🆕 신규 디렉터리: bench_results/sweep_20260429_154331
이미 완료: 0 / 32
y pickled → bench_results/sweep_20260429_154331/y.pkl  (len=26279)

[001/32] rustima  seasonal=False(s=0) stepwise=True trend=n
  ▶ attempt 1/1
    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(5, 0, 4) aic=176435.1018 loglik=-88207.5509 time=7.4984s peak=182.89MB

[002/32] pmdarima  seasonal=False(s=0) stepwise=True trend=n
  ▶ attempt 1/1
    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(2, 1, 3) aic=177662.5151 loglik=-88825.2576 time=27.8825s peak=1073.39MB

[003/32] rustima  seasonal=False(s=0) stepwise=True trend=c
  ▶ attempt 1/1
    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(2, 0, 5) aic=180156.9791 loglik=-90069.4896 time=6.2102s peak=182.09MB

[004/32] pmdarima  seasonal=False(s=0) stepwise=True trend=c
  ▶ attempt 1/1
    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(2, 1, 3) aic=177664.5130 loglik=-88825.2565 time=51.1590s peak=1340.20MB

[005/32] rustima  seasonal

python(11325) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(3, 0, 2) aic=162665.0419 loglik=-81321.5210 time=642.2387s peak=290.28MB

[020/32] pmdarima  seasonal=True(s=24) stepwise=True trend=c
  ▶ attempt 1/1


python(11957) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


    → crashed: 
  ▣ final [crashed] attempts=1 history=crashed

[021/32] rustima  seasonal=True(s=24) stepwise=True trend=t
  ▶ attempt 1/1


...

    → crashed: 
  ▣ final [crashed] attempts=1 history=crashed

[031/32] rustima  seasonal=True(s=24) stepwise=False trend=ct
  ▶ attempt 1/1


python(20350) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


    → ok
  ▣ final [ok] attempts=1 history=ok
    order=(4, 0, 5) aic=162863.9106 loglik=-81415.9553 time=354.9628s peak=933.12MB

[032/32] pmdarima  seasonal=True(s=24) stepwise=False trend=ct
  ▶ attempt 1/1


python(20695) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


    → crashed: 
  ▣ final [crashed] attempts=1 history=crashed

✅ 세션 종료 — 누적 완료 32/32, 이번 세션 198.3분 소요
   CSV : bench_results/sweep_20260429_154331/sweep_results.csv
   로그 : bench_results/sweep_20260429_154331/logs
   메모리 : bench_results/sweep_20260429_154331/mem_samples


## 요약 — rustima vs pmdarima 비교 표

- Table 1: AIC + 속도 + 피크 메모리 비교
- Table 2: 선택된 order + loglikelihood 비교

In [18]:
# ======================================================================
# 요약 테이블 — rustima vs pmdarima 비교
# ======================================================================
import pandas as pd
import numpy as np
from pathlib import Path

csvs = sorted(Path("bench_results").glob("sweep_*/sweep_results.csv"))
assert csvs, "sweep-run 먼저 실행하세요."
latest = csvs[-1]
out_dir = latest.parent
print(f"Source : {latest}")


def _reason(status, error):
    if status == "ok":
        return ""
    el = (error or "").lower()
    for kw, tag in [("timeout", "TIMEOUT"), ("memory", "OOM"),
                     ("overflow", "OVERFLOW"), ("converge", "NOCONV"),
                     ("singular", "SINGULAR"), ("linalg", "LINALG"),
                     ("nan", "NAN")]:
        if kw in el:
            return tag
    if status == "timeout":  return "TIMEOUT"
    if status == "crashed":  return "CRASH"
    return "FAIL" if status != "ok" else ""


# ---- 로드 ------------------------------------------------------------
df = pd.read_csv(latest)
for c in ("time_s", "mem_baseline_mb", "mem_peak_mb", "mem_delta_mb",
          "ic_value", "loglik_own"):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
df["error"] = df["error"].fillna("") if "error" in df.columns else ""
df["reason"] = df.apply(lambda r: _reason(r.get("status", ""), r.get("error", "")), axis=1)

key_cols = ["idx", "seasonal", "s", "stepwise", "trend"]


def _lib_frame(lib, prefix):
    value_cols = [c for c in ("order", "seasonal_order", "ic_value", "time_s",
                               "mem_peak_mb", "loglik_own", "status", "reason")
                  if c in df.columns]
    sub = df[df.library == lib][key_cols + value_cols].copy()
    rename = {c: f"{prefix}_{c}" for c in value_cols}
    return sub.rename(columns=rename)


merged = _lib_frame("rustima", "rs").merge(
    _lib_frame("pmdarima", "pm"), on=key_cols, how="outer"
).sort_values("idx").reset_index(drop=True)


def _full_order(o, so, reason=""):
    if reason: return f"[{reason}]"
    if pd.isna(o) or str(o).strip() == "": return ""
    so_clean = str(so).replace(" ", "") if pd.notna(so) else ""
    if so_clean in ("(0,0,0,0)", "(0,0,0)", "()", ""):
        return str(o)
    return f"{o}{so}"


# ---- Table 1 — AIC + 속도 + 피크 메모리 ----------------------------
t1 = merged[key_cols + ["rs_ic_value", "pm_ic_value", "rs_time_s", "pm_time_s",
                         "rs_mem_peak_mb", "pm_mem_peak_mb",
                         "rs_reason", "pm_reason"]].copy()
t1["delta_aic"] = t1["pm_ic_value"]    - t1["rs_ic_value"]
t1["speedup"]   = t1["pm_time_s"]      / t1["rs_time_s"]
t1["mem_ratio"] = t1["pm_mem_peak_mb"] / t1["rs_mem_peak_mb"]

# 포맷
def _f(v, fmt, reason=""):
    if reason: return f"[{reason}]"
    return fmt.format(v) if pd.notna(v) else ""

for col, fmt, rcol in [
    ("rs_ic_value",    "{:.2f}",  "rs_reason"),
    ("pm_ic_value",    "{:.2f}",  "pm_reason"),
    ("delta_aic",      "{:+.2f}", None),
    ("rs_time_s",      "{:.3f}",  "rs_reason"),
    ("pm_time_s",      "{:.3f}",  "pm_reason"),
    ("speedup",        "{:.2f}x", None),
    ("rs_mem_peak_mb", "{:.1f}",  "rs_reason"),
    ("pm_mem_peak_mb", "{:.1f}",  "pm_reason"),
    ("mem_ratio",      "{:.2f}x", None),
]:
    if rcol:
        t1[col] = [_f(v, fmt, r) for v, r in zip(t1[col], t1[rcol])]
    else:
        t1[col] = t1[col].apply(lambda v: _f(v, fmt))

t1_out = t1[key_cols + [
    "rs_ic_value", "pm_ic_value", "delta_aic",
    "rs_time_s", "pm_time_s", "speedup",
    "rs_mem_peak_mb", "pm_mem_peak_mb", "mem_ratio",
]].rename(columns={
    "rs_ic_value": "rs_aic", "pm_ic_value": "pm_aic",
    "rs_time_s": "rs_time(s)", "pm_time_s": "pm_time(s)",
    "rs_mem_peak_mb": "rs_mem(MB)", "pm_mem_peak_mb": "pm_mem(MB)",
    "speedup": "speedup(x)", "mem_ratio": "mem_ratio(x)",
})

# ---- Table 2 — order + loglik ----------------------------------------
t2 = merged[key_cols + ["rs_order", "pm_order",
                         "rs_seasonal_order", "pm_seasonal_order",
                         "rs_loglik_own", "pm_loglik_own",
                         "rs_reason", "pm_reason"]].copy()
t2["rs_full_order"] = [_full_order(o, so, r) for o, so, r in
                       zip(t2["rs_order"], t2["rs_seasonal_order"], t2["rs_reason"])]
t2["pm_full_order"] = [_full_order(o, so, r) for o, so, r in
                       zip(t2["pm_order"], t2["pm_seasonal_order"], t2["pm_reason"])]
t2["same_order"]    = [(r == p) and r != "" and not r.startswith("[")
                       for r, p in zip(t2["rs_full_order"], t2["pm_full_order"])]
t2["delta_loglik"]  = t2["rs_loglik_own"] - t2["pm_loglik_own"]

for col, fmt, rcol in [
    ("rs_loglik_own", "{:.2f}", "rs_reason"),
    ("pm_loglik_own", "{:.2f}", "pm_reason"),
    ("delta_loglik",  "{:+.2f}", None),
]:
    if rcol:
        t2[col] = [_f(v, fmt, r) for v, r in zip(t2[col], t2[rcol])]
    else:
        t2[col] = t2[col].apply(lambda v: _f(v, fmt))

t2_out = t2[key_cols + [
    "rs_full_order", "pm_full_order", "same_order",
    "rs_loglik_own", "pm_loglik_own", "delta_loglik",
]].rename(columns={
    "rs_full_order": "rs_order", "pm_full_order": "pm_order",
    "rs_loglik_own": "rs_loglik", "pm_loglik_own": "pm_loglik",
})

# ---- 저장 + 출력 -----------------------------------------------------
t1_out.to_csv(out_dir / "table1_aic_speed_mem.csv", index=False)
t2_out.to_csv(out_dir / "table2_order_loglik.csv", index=False)

with pd.option_context("display.max_rows", 50, "display.width", 260,
                        "display.max_colwidth", 30):
    print("\n" + "=" * 100)
    print("Table 1 — AIC + 속도 + 피크 메모리   (delta = pm - rs,  speedup/mem_ratio = pm / rs)")
    print("=" * 100)
    print(t1_out.to_string(index=False))

    print("\n" + "=" * 100)
    print("Table 2 — 선택된 order + loglikelihood   (delta = rs - pm)")
    print("=" * 100)
    print(t2_out.to_string(index=False))

# 실패 런 요약
fail_df = df[df["reason"] != ""]
if len(fail_df):
    print(f"\n⚠️  실패 런 {len(fail_df)}건:")
    print(fail_df[["idx", "library", "seasonal", "stepwise", "trend",
                    "status", "reason"]].to_string(index=False))
else:
    print("\n✨ 실패 런 없음.")

Source : bench_results/sweep_20260429_154331/sweep_results.csv

Table 1 — AIC + 속도 + 피크 메모리   (delta = pm - rs,  speedup/mem_ratio = pm / rs)
 idx  seasonal  s  stepwise trend    rs_aic    pm_aic delta_aic rs_time(s) pm_time(s) speedup(x) rs_mem(MB) pm_mem(MB) mem_ratio(x)
   0     False  0      True     n 176435.10 177662.52  +1227.41      7.498     27.883      3.72x      182.9     1073.4        5.87x
   1     False  0      True     c 180156.98 177664.51  -2492.47      6.210     51.159      8.24x      182.1     1340.2        7.36x
   2     False  0      True     t 181966.43 179503.05  -2463.39      3.289     26.667      8.11x      180.1     1054.3        5.85x
   3     False  0      True    ct 182920.70 179505.37  -3415.34      9.966     28.378      2.85x      180.2     1060.5        5.88x
   4     False  0     False     n 176435.10 177662.52  +1227.41      3.605     19.847      5.51x      197.1     1755.1        8.91x
   5     False  0     False     c 180156.98 177664.51  -2492.47   

## 시각화 — 메모리 타임라인 + 막대그래프 + 비교 차트

In [19]:
# ======================================================================
# 시각화 — 막���그래프 + 메모리 타임라인 그리드 + 조합별 개별 타임��인
# ======================================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# macOS 한글 폰트
plt.rcParams["font.family"] = ["AppleGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

csvs = sorted(Path("bench_results").glob("sweep_*/sweep_results.csv"))
assert csvs, "먼저 sweep-run 셀을 실행하세��."
latest = csvs[-1]
out_dir = latest.parent
plot_dir = out_dir / "plots"
tl_dir = plot_dir / "timelines"
plot_dir.mkdir(exist_ok=True)
tl_dir.mkdir(exist_ok=True)
print(f"Source : {latest}")
print(f"Plots  : {plot_dir}")

df = pd.read_csv(latest)
for c in ["time_s", "mem_baseline_mb", "mem_peak_mb", "mem_delta_mb"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

def _make_combo(r):
    return f"s{r.s}_sw{int(r.stepwise)}_{r.trend}"

df["combo"] = df.apply(_make_combo, axis=1)
ok = df[df.status == "ok"].copy()
ok["combo"] = ok.apply(_make_combo, axis=1)

LIB_COLORS = {"rustima": "#5bc0de", "pmdarima": "#d9534f"}
all_combos = sorted(df["combo"].unique())

# ------------------------------------------------------------------
# 1. 막대그래프 — 조합별 실행시간
# ------------------------------------------------------------------
if len(ok):
    pivot_t = ok.pivot_table(index="combo", columns="library", values="time_s")
    pivot_t = pivot_t.reindex(all_combos)
    cols_present = [c for c in ["rustima", "pmdarima"] if c in pivot_t.columns]
    colors = [LIB_COLORS[c] for c in cols_present]

    fig, ax = plt.subplots(figsize=(max(10, 0.6 * len(all_combos) + 4), 5.5))
    pivot_t[cols_present].plot(kind="bar", ax=ax, color=colors)
    ax.legend(framealpha=0.4)
    ax.set_ylabel("time (s)")
    ax.set_title("조합별 실행 시간 — rustima vs pmdarima")
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=60, labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(plot_dir / "bars_time.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("  saved bars_time.png")

# ------------------------------------------------------------------
# 2. 막대그래프 — 조합별 피크 메모리
# ------------------------------------------------------------------
if len(df):
    pivot_m = df.pivot_table(index="combo", columns="library",
                              values="mem_peak_mb", aggfunc="first")
    pivot_m = pivot_m.reindex(all_combos)
    cols_present = [c for c in ["rustima", "pmdarima"] if c in pivot_m.columns]
    colors = [LIB_COLORS[c] for c in cols_present]

    fig, ax = plt.subplots(figsize=(max(10, 0.6 * len(all_combos) + 4), 5.5))
    pivot_m[cols_present].plot(kind="bar", ax=ax, color=colors)
    ax.legend(framealpha=0.4)
    ax.set_ylabel("peak RSS (MB)")
    ax.set_title("조합별 피크 메���리 — rustima vs pmdarima")
    ax.set_xlabel("")
    ax.tick_params(axis="x", labelrotation=60, labelsize=8)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(plot_dir / "bars_mem.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("  saved bars_mem.png")

# ------------------------------------------------------------------
# 3. 그룹별 박스플롯 — stepwise별 / trend별
# ------------------------------------------------------------------
def _grouped_boxplot(group_col, fname_stem, title_label):
    cats = sorted(ok[group_col].dropna().unique().tolist(), key=lambda x: str(x))
    if not cats:
        return
    fig, axes = plt.subplots(1, 2, figsize=(max(8, 1.8 * len(cats) + 4), 5))
    width = 0.36
    x = np.arange(len(cats))
    for ax_idx, (metric, ylabel, ax_title) in enumerate([
        ("mem_peak_mb", "peak RSS (MB)", f"피크 메모리 — {title_label}별"),
        ("time_s",      "time (s)",      f"실행 시간 — {title_label}별"),
    ]):
        ax = axes[ax_idx]
        for i, lib in enumerate(["rustima", "pmdarima"]):
            data = [ok[(ok.library == lib) & (ok[group_col] == c)][metric]
                    .dropna().values for c in cats]
            offset = (i - 0.5) * width
            bp = ax.boxplot(
                data, positions=x + offset, widths=width * 0.9,
                patch_artist=True, showfliers=True,
                medianprops=dict(color="black", linewidth=1.4),
                flierprops=dict(marker="o", markersize=3, alpha=0.5),
            )
            for box in bp["boxes"]:
                box.set(facecolor=LIB_COLORS[lib], alpha=0.6, edgecolor="black")
            ax.plot([], [], color=LIB_COLORS[lib], lw=8, alpha=0.6, label=lib)
        ax.set_xticks(x)
        ax.set_xticklabels([str(c) for c in cats])
        ax.set_xlabel(title_label)
        ax.set_ylabel(ylabel)
        ax.set_title(ax_title)
        ax.legend(framealpha=0.4)
        ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(plot_dir / f"compare_{fname_stem}.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  saved compare_{fname_stem}.png")

_grouped_boxplot("stepwise", "by_stepwise", "stepwise")
_grouped_boxplot("trend",    "by_trend",    "trend")

# ------------------------------------------------------------------
# 4. 메모리 타임라인 — 그리드 + 개별
# ------------------------------------------------------------------
def _load_samples(mem_file):
    p = out_dir / mem_file
    if not p.exists():
        return [], []
    d = json.loads(p.read_text(encoding="utf-8"))
    s = d.get("samples") or []
    if not s:
        return [], []
    t, m = zip(*s)
    return list(t), list(m)

mem_combos = sorted(df.loc[df["mem_file"].notna(), "combo"].unique())
n = len(mem_combos)

if n:
    ncols = 4
    nrows = int(np.ceil(n / ncols))
    fig_g, axes_g = plt.subplots(nrows, ncols,
                                 figsize=(ncols * 3.8, nrows * 2.8),
                                 sharex=False)
    axes_flat = np.atleast_1d(axes_g).flatten()
    for ax in axes_flat[n:]:
        ax.axis("off")
    for i, combo in enumerate(mem_combos):
        ax = axes_flat[i]
        for lib, color in [("rustima", "#5bc0de"), ("pmdarima", "#d9534f")]:
            row = df[(df.combo == combo) & (df.library == lib)]
            if row.empty:
                continue
            t, m = _load_samples(row.iloc[0]["mem_file"])
            if t:
                ax.plot(t, m, color=color, label=lib, lw=1.1)
        ax.set_title(combo, fontsize=9)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.25)
        if i == 0:
            ax.legend(fontsize=7, framealpha=0.4)
    fig_g.suptitle("메모리 타임라인 — 전체 조합 (파주 열수요)", fontsize=13, y=1.002)
    fig_g.supxlabel("elapsed (s)", fontsize=9)
    fig_g.supylabel("RSS (MB)", fontsize=9)
    fig_g.tight_layout()
    fig_g.savefig(plot_dir / "timelines_grid.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig_g)
    print("  saved timelines_grid.png")

    # 개별 타임라인
    for combo in mem_combos:
        fig, ax = plt.subplots(figsize=(9, 3.6))
        for lib, color in [("rustima", "#5bc0de"), ("pmdarima", "#d9534f")]:
            row = df[(df.combo == combo) & (df.library == lib)]
            if row.empty:
                continue
            t, m = _load_samples(row.iloc[0]["mem_file"])
            if t:
                peak = row.iloc[0]["mem_peak_mb"]
                dur = row.iloc[0]["time_s"]
                status = row.iloc[0]["status"]
                tag = "" if status == "ok" else f" [{status.upper()}]"
                ax.plot(t, m, color=color,
                        label=f"{lib}{tag}  peak={peak:.1f}MB  time={dur:.2f}s",
                        lw=1.3)
        ax.set_xlabel("elapsed (s)")
        ax.set_ylabel("RSS (MB)")
        ax.set_title(f"메모��� 타임라인 — {combo}")
        ax.legend(framealpha=0.4)
        ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(tl_dir / f"timeline_{combo}.png", dpi=100, bbox_inches="tight")
        plt.close(fig)
    print(f"  saved {len(mem_combos)} per-combo timelines → {tl_dir}")

print("\n✅ 시각화 완료")

Source : bench_results/sweep_20260429_154331/sweep_results.csv
Plots  : bench_results/sweep_20260429_154331/plots
  saved bars_time.png
  saved bars_mem.png
  saved compare_by_stepwise.png
  saved compare_by_trend.png
  saved timelines_grid.png
  saved 16 per-combo timelines → bench_results/sweep_20260429_154331/plots/timelines

✅ 시각화 완료
